In [28]:
import pandas as pd

In [29]:
data_path = "../player-data/Master.csv"
df = pd.read_csv(data_path)

In [30]:
print(df.columns)

Index(['playerID', 'birthYear', 'birthMonth', 'birthDay', 'birthCountry',
       'birthState', 'birthCity', 'deathYear', 'deathMonth', 'deathDay',
       'deathCountry', 'deathState', 'deathCity', 'nameFirst', 'nameLast',
       'nameGiven', 'weight', 'height', 'bats', 'throws', 'debut', 'finalGame',
       'retroID', 'bbrefID'],
      dtype='object')


In [31]:
multiples_count = (df['nameGiven'].value_counts() > 1).sum()
print(multiples_count)

2125


In [32]:
df_unique = df[['playerID', 'nameGiven', 'nameFirst', 'nameLast']].copy()
df_unique

,playerID,nameGiven,nameFirst,nameLast
0,aardsda01,David Allan,David,Aardsma
1,aaronha01,Henry Louis,Hank,Aaron
2,aaronto01,Tommie Lee,Tommie,Aaron
3,aasedo01,Donald William,Don,Aase
4,abadan01,Fausto Andres,Andy,Abad
...,...,...,...,...
18841,zupofr01,Frank Joseph,Frank,Zupo
18842,zuvelpa01,Paul,Paul,Zuvella
18843,zuverge01,George,George,Zuverink
18844,zwilldu01,Edward Harrison,Dutch,Zwilling


In [33]:
# check for null values in given name column
df_unique['nameGiven'].isna().sum()

np.int64(39)

In [34]:
df_unique['nameFirst'].isna().sum()

np.int64(39)

In [35]:
df_unique['nameLast'].isna().sum()

np.int64(0)

In [36]:
# all missing given names also have missing first names - delete these rows
df_unique[df_unique['nameFirst'].isna() & df_unique['nameGiven'].isna()].shape

(39, 4)

In [37]:
df_unique.dropna(subset=['nameGiven'], inplace=True)
df_unique.shape

(18807, 4)

In [38]:
mask = df_unique['nameGiven'].apply(lambda x: ' ' not in x)
names_to_fix = df_unique.loc[mask, 'nameGiven'].to_dict()
print(len(names_to_fix))

1633


In [39]:
bad_rows = []

for idx, name in names_to_fix.items():
    curr_row = df_unique.loc[idx]
    first = curr_row['nameFirst']
    last = curr_row['nameLast']

    if name == first:
        df_unique.at[idx, 'nameGiven'] = f"{first} {last}" # add last name to given name
    elif name == last:
        df_unique.at[idx, 'nameGiven'] = f"{first} {last}" # add first name to given name
    else:
        bad_rows.append(idx) # needs manual fixing
    
print(len(bad_rows))

759


In [40]:
df_unique = df_unique.drop(index=bad_rows)
df_unique.shape

(18048, 4)

In [41]:
df_unique['nameGiven'].value_counts()

nameGiven
John Joseph       74
William Henry     53
William Joseph    48
Michael Joseph    42
John William      40
                  ..
Mauro Paul         1
Billy Cordell      1
Rodney Blaine      1
Alfons Francis     1
Anthony Aaron      1
Name: count, Length: 12634, dtype: int64

In [42]:
name_counts = df_unique['nameGiven'].value_counts()

# Split into two DataFrames
df_duplicates = df_unique[df_unique['nameGiven'].isin(name_counts[name_counts > 1].index)]
df_nonduplicates = df_unique[df_unique['nameGiven'].isin(name_counts[name_counts == 1].index)]

print("Duplicates:", len(df_duplicates))
print("Non-duplicates:", len(df_nonduplicates))

Duplicates: 7326
Non-duplicates: 10722


In [43]:
players = (
    df_nonduplicates[['nameGiven']]
    .sort_values('nameGiven')
    .reset_index(drop=True)
)

# players.to_csv('players_nonduplicates.csv', index=False)
# print(f"{len(players)} unique players written to players_nonduplicates.csv")

# players.iloc[:2000].to_csv('players_batchA.csv', index=False)
# players.iloc[2000:4000].to_csv('players_batchB.csv', index=False)
# players.iloc[4000:7000].to_csv('players_batchC.csv', index=False)
# players.iloc[7000:].to_csv('players_batchD.csv', index=False)

In [44]:
df[['nameGiven', 'nameFirst', 'nameLast']]


,nameGiven,nameFirst,nameLast
0,David Allan,David,Aardsma
1,Henry Louis,Hank,Aaron
2,Tommie Lee,Tommie,Aaron
3,Donald William,Don,Aase
4,Fausto Andres,Andy,Abad
...,...,...,...
18841,Frank Joseph,Frank,Zupo
18842,Paul,Paul,Zuvella
18843,George,George,Zuverink
18844,Edward Harrison,Dutch,Zwilling


In [45]:
import re
import unicodedata
from itertools import product

In [46]:
def strip_accents(s: str) -> str:
    if pd.isna(s): return ""
    s = str(s)
    return ''.join(c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c))

def norm_space(s: str) -> str:
    return re.sub(r"\s+", " ", str(s or "")).strip()

def titleish(s: str) -> str:
    return norm_space(s).title()

def given_prefixes(given: str) -> list[str]:
    # "Manuel Arturo" -> ["Manuel", "Manuel Arturo"]
    g = norm_space(given)
    if not g: return []
    toks, out, cur = g.split(), [], []
    for t in toks:
        cur.append(t)
        out.append(" ".join(cur))
    return out

def last_variants(last: str) -> list[str]:
    last = norm_space(last)
    if not last: return []
    la = strip_accents(last)
    variants = {last, la}

    # hyphen -> space
    if "-" in last: variants.add(last.replace("-", " "))
    if "-" in la:   variants.add(la.replace("-", " "))

    # apostrophes (O'Neill → O Neill / ONeill) and curly apostrophes
    for base in (last, la):
        b = base.replace("’", "'")
        if "'" in b:
            variants.add(b.replace("'", " "))
            variants.add(b.replace("'", ""))

    return sorted({norm_space(v) for v in variants if norm_space(v)})

def first_side_options(name_first: str, name_given: str) -> list[str]:
    opts = set()

    nf = norm_space(name_first)
    if nf:
        opts.add(nf); opts.add(strip_accents(nf))

    # progressive prefixes from nameGiven
    ng = norm_space(name_given)
    for seq in given_prefixes(ng):
        opts.add(seq); opts.add(strip_accents(seq))

    # remove 1-char fragments (avoid “A”)
    opts = {o for o in (norm_space(x) for x in opts) if len(o) > 1}
    return sorted(opts, key=lambda s: (len(s), s.lower()))

def make_keywords_for_row(row, max_per_player: int = 25) -> list[str]:
    last = norm_space(row.get("nameLast", ""))
    if not last:
        return []  # need a last name for stable phrases

    first_opts = first_side_options(row.get("nameFirst", ""), row.get("nameGiven", ""))
    last_opts  = last_variants(last)

    combos = []
    # Full first-ish + last
    for f, l in product(first_opts, last_opts):
        combos.append(titleish(f + " " + l))

    # First-initial + last (e.g., "M. Machado")
    nf = norm_space(row.get("nameFirst", ""))
    if nf:
        fi = nf[:1] + "."
        if len(fi) == 2:
            for l in last_opts:
                combos.append(titleish(fi + " " + l))

    # Deduplicate & cap
    uniq = sorted(set(combos), key=lambda s: (len(s), s.lower()))
    return uniq[:max_per_player]


In [47]:
# Ensure expected columns exist
for col in ["nameFirst", "nameLast", "nameGiven"]:
    if col not in df.columns:
        df[col] = ""

# Label to track the player in output (prefer "First Last"; fallback to "Given Last")
df["player_label"] = (
    (df["nameFirst"].fillna("") + " " + df["nameLast"].fillna(""))
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)
empty_mask = df["player_label"].eq("")
df.loc[empty_mask, "player_label"] = (
    (df.loc[empty_mask, "nameGiven"].fillna("") + " " + df.loc[empty_mask, "nameLast"].fillna(""))
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Generate keyword lists
df["keywords"] = df.apply(lambda r: make_keywords_for_row(r, max_per_player=25), axis=1)

# Explode to (player_label, keyword)
kw_df = (
    df.loc[df["keywords"].str.len() > 0, ["player_label", "keywords"]]
      .explode("keywords")
      .rename(columns={"keywords": "keyword"})
      .drop_duplicates()
      .reset_index(drop=True)
)

print(kw_df.head(10))
print(f"Players with keywords: {kw_df['player_label'].nunique()} | total keyword rows: {len(kw_df)}")

kw_df.to_csv("player_keywords_all.csv", index=False)
print("Wrote player_keywords_all.csv")

    player_label              keyword
0  David Aardsma           D. Aardsma
1  David Aardsma        David Aardsma
2  David Aardsma  David Allan Aardsma
3     Hank Aaron             H. Aaron
4     Hank Aaron           Hank Aaron
5     Hank Aaron          Henry Aaron
6     Hank Aaron    Henry Louis Aaron
7   Tommie Aaron             T. Aaron
8   Tommie Aaron         Tommie Aaron
9   Tommie Aaron     Tommie Lee Aaron
Players with keywords: 18228 | total keyword rows: 66117
Wrote player_keywords_all.csv


In [49]:
kw_df[kw_df["player_label"] == "Bob Alexander"]

,player_label,keyword
575,Bob Alexander,B. Alexander
576,Bob Alexander,Bob Alexander
577,Bob Alexander,Robert Alexander
578,Bob Alexander,Robert Somerville Alexander
